# Types of estimators in scikit-learn

It's important to clarify certain terms used in scikit-learn:

- **Estimators**: any object that can estimate parameters based on a dataset is called an estimator. The estimation itself is performed through the **`fit()`** method.

    - **Transformers**: estimators that can also transform data using the **`transform()`** method. For example, `SimpleImputer` is a transformer: it estimates values with `fit()` and imputes them with `transform()`.
        - Scalers: transformers that scale data.
        - Imputers: transformers that impute missing values.
        - Encoders: transformers that encode categorical variables.
        - Dimensionality reducers: transformers that reduce the number of variables.
        - ...

    - **Predictors**: those estimators that are capable of making predictions based on a dataset. For example, the linear regression model is a predictor: it estimates its parameters with `fit()` and makes predictions with **`predict()`**.
        - Classifiers: predictors that predict categorical labels.
        - Regressors: predictors that predict continuous values.
        - Clusterers: predictors that group data into clusters.
        - ...

The term "predictor" can be confusing as it is also used, in general, to refer to the *features* or independent variables of a model, and sometimes only for those variables that are indeed **predictive**, leaving out those characteristics that do not have predictive capacity.

Also, the term "transformer" should not be confused with the popular "transformer" neural network architecture, which is the basis on which language models like GPT are built.

## Transformers: `fit()` and `transform()`

A transformer *learns* something from the data in `fit()` and *applies* it in `transform()`. `SimpleImputer` learns the median of each column; the learned values are stored in `statistics_`.

In [ ]:
import numpy as np
from sklearn.impute import SimpleImputer

X = np.array([[1.0, 10.0],
              [2.0, np.nan],
              [np.nan, 30.0],
              [4.0, 40.0]])

imputer = SimpleImputer(strategy="median")
imputer.fit(X)  # learns the median of each column from the data
imputer.statistics_

array([ 2., 30.])

In [ ]:
imputer.transform(X)  # applies what was learned: fills each gap with its column's median

array([[ 1., 10.],
       [ 2., 30.],
       [ 2., 30.],
       [ 4., 40.]])

Separating the two steps is what makes the transformer reusable on new data: the medians learned from the training set are the ones applied to the test set, or to any future input, without recomputing anything (see [Data leakage](data_leakage.ipynb) for why this matters).

In [ ]:
X_new = np.array([[np.nan, np.nan]])
imputer.transform(X_new)  # the medians of the ORIGINAL data are used, not those of X_new

array([[ 2., 30.]])

## Predictors: `fit()` and `predict()`

A predictor learns its **parameters** in `fit()` and uses them in `predict()`. For a linear regression the parameters are the coefficients and the intercept.

In [ ]:
from sklearn.linear_model import LinearRegression

X_lin = np.array([[1.0], [2.0], [3.0], [4.0]])
y_lin = np.array([3.0, 5.0, 7.0, 9.0])  # y = 2x + 1

lin_reg = LinearRegression()
lin_reg.fit(X_lin, y_lin)
lin_reg.coef_, lin_reg.intercept_

(array([2.]), np.float64(1.0))

In [ ]:
lin_reg.predict(np.array([[5.0], [6.0]]))

array([11., 13.])

## Conventions shared by all estimators

- **Hyperparameters** are set in the constructor and never change during `fit()`: `SimpleImputer(strategy="median")`, `LinearRegression(fit_intercept=True)`. They can be read with `get_params()` and changed with `set_params()`.
- **Learned attributes** end with an underscore (`statistics_`, `coef_`, `intercept_`, `categories_`) and only exist after `fit()`. Accessing them before fitting raises an exception.
- `fit()` returns the estimator itself, so calls can be chained: `SimpleImputer().fit(X).transform(X)`.
- Transformers also have **`fit_transform()`**, which does both steps in one call (sometimes more efficiently). Predictors have **`score()`**, which returns a default metric (R² for regressors, accuracy for classifiers).

In [ ]:
imputer.get_params()

{'add_indicator': False,
 'copy': True,
 'fill_value': None,
 'keep_empty_features': False,
 'missing_values': nan,
 'strategy': 'median'}

In [ ]:
imputer.set_params(strategy="mean")
imputer.fit(X).statistics_  # after refitting, the learned attribute reflects the new hyperparameter

array([ 2.33333333, 26.66666667])

In [ ]:
from sklearn.exceptions import NotFittedError

try:
    LinearRegression().predict(X_lin)
except NotFittedError as e:
    print("NotFittedError:", e)

NotFittedError: This LinearRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


This uniform interface is what allows estimators to be combined into [pipelines](pipelines.ipynb) and to be tuned with the same tools ([Hyperparameter search](hyperparameter_search.ipynb)), and it is the contract to respect when writing [custom transformers](custom_transformers.ipynb). The full design is described in the [scikit-learn developer guide](https://scikit-learn.org/stable/developers/develop.html).